# 04 · The graph you wrote

Chapter 03 let one call decide what to look at next. Here the looking is decided in
advance: five lenses, written down, run in parallel, merged.

> **You'll learn**
> - Fan out to several reasoners at once with `asyncio.gather` over `router.app.call`
> - Read a DAG whose shape is a property of your code, not of the input
> - See what a fixed process costs you when the incident is not the one you planned for

`incidents/lenses.md` catalogues 26 lenses an SRE runs. A lens is a standing question
plus the discipline of answering it. Five of them run on almost every incident.

In [1]:
import sys, json, requests
sys.path.insert(0, "../lib")
import dag

SERVER = "http://localhost:8080"

def run(reasoner, incident_id):
    """Drive the node through the control plane. Never `await app.call` from a notebook."""
    r = requests.post(f"{SERVER}/api/v1/execute/blast-radius.{reasoner}",
                      json={"input": {"incident_id": incident_id}}, timeout=600)
    r.raise_for_status()
    d = r.json()
    print(f"{incident_id}  {d['status']}  {d['duration_ms']/1000:.0f}s  run={d['run_id']}")
    return d

dag.print_nodes()

   (10 unrelated agent(s) on this control plane, not shown)
   1 agent(s) registered:
     - blast-radius [active] 24 reasoner(s) @ http://127.0.0.1:8002
   NOTE: registrations outlive the process. health_status can lie;
         dag.node_alive(<agent_id>) pings the node itself.


So `r04` hardcodes those five. Each is its own reasoner, each reads only the artifacts
its question needs, each returns the same small `LensReport`.

In [2]:
src = open("../node/rungs/r04.py").read()
block = src[src.index("LENSES = {"):src.index("\nasync def _lens")]
print(block)


LENSES = {
    "timeline": (
        "When did this truly start? Compare first-bad timestamps against the alert time and "
        "against every change time. The alert time is never the incident time. A candidate "
        "cause that postdates the symptom is not the cause.",
        ["alert", "logs", "metrics", "deploys"],
    ),
    "change_correlation": (
        "What changed inside the window, and does the diff touch the failing path? Do not blame "
        "a change for being the newest, and do not dismiss a one-line change for being small.",
        ["alert", "deploys", "logs"],
    ),
    "blast_scope": (
        "Who is affected and, more importantly, who is not? Look for a clean split along one "
        "dimension: per-pod, per-region, per-tenant, per-currency, per-endpoint, per-caller. "
        "The negative half is the powerful half.",
        ["alert", "metrics", "topology", "logs"],
    ),
    "dependency_health": (
        "Are our downstreams healthy? Look for one de

`diagnose` does no reasoning of its own. It gathers the five, then hands them to a
sixth call that merges them.

In [3]:
import re
src = open("../node/rungs/r04.py").read()
print(src[src.index('@router.reasoner(tags=["entry"])'):])

@router.reasoner(tags=["entry"])
async def diagnose(incident_id: str, model: str | None = None) -> Diagnosis:
    """Fan out to five fixed lenses in parallel, then merge."""
    reports = await asyncio.gather(
        *(
            router.app.call(f"{NODE_ID}.r04_lens_{name}", incident_id=incident_id, model=model)
            for name in LENSES
        )
    )
    return await router.app.call(
        f"{NODE_ID}.r04_synthesize", incident_id=incident_id, reports=list(reports), model=model
    )



Run it on two incidents that have nothing in common. `inc-008` is a memory leak that
grew for nine days; `inc-011` is an auth failure on one pod.

In [4]:
a = run("r04_diagnose", "inc-008")
b = run("r04_diagnose", "inc-011")

inc-008  succeeded  28s  run=run_20260820_122838_rm9w8042


inc-011  succeeded  42s  run=run_20260820_122906_2ulmx8xu


## The two graphs

In [5]:
dag.render_two(a["run_id"], b["run_id"], labels=("inc-008 · memory leak", "inc-011 · auth failures"))

```mermaid
flowchart LR
  subgraph ag["inc-008 · memory leak — 7 exec · depth 2 · fan-out 6"]
  direction TD
    a0["r04_diagnose<br/><small>✓ succeeded · 27.5s</small>"]
    a1["r04_lens_timeline<br/><small>✓ succeeded · 13.1s</small>"]
    a2["r04_lens_change_correlation<br/><small>✓ succeeded · 8.1s</small>"]
    a3["r04_lens_dependency_health<br/><small>✓ succeeded · 5.9s</small>"]
    a4["r04_lens_blast_scope<br/><small>✓ succeeded · 9.1s</small>"]
    a5["r04_lens_resource_contention<br/><small>✓ succeeded · 7.4s</small>"]
    a6["r04_synthesize<br/><small>✓ succeeded · 13.7s</small>"]
    a0 --> a1
    a0 --> a2
    a0 --> a3
    a0 --> a4
    a0 --> a5
    a0 --> a6
    class a0,a1,a2,a3,a4,a5,a6 ok;
  end
  subgraph bg["inc-011 · auth failures — 7 exec · depth 2 · fan-out 6"]
  direction TD
    b0["r04_diagnose<br/><small>✓ succeeded · 41.9s</small>"]
    b1["r04_lens_timeline<br/><small>✓ succeeded · 27.0s</small>"]
    b2["r04_lens_change_correlation<br/><small>✓ succeeded · 12.3s</small>"]
    b3["r04_lens_blast_scope<br/><small>✓ succeeded · 6.3s</small>"]
    b4["r04_lens_resource_contention<br/><small>✓ succeeded · 6.0s</small>"]
    b5["r04_lens_dependency_health<br/><small>✓ succeeded · 7.9s</small>"]
    b6["r04_synthesize<br/><small>✓ succeeded · 14.4s</small>"]
    b0 --> b1
    b0 --> b2
    b0 --> b3
    b0 --> b4
    b0 --> b5
    b0 --> b6
    class b0,b1,b2,b3,b4,b5,b6 ok;
  end
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

They are identical. Same seven nodes, same fan-out of five, same merge — because the
graph is a property of `r04.py`, and `r04.py` did not read the incident.

In [6]:
def shape(d):
    ex = dag.fetch_run(d["run_id"])["executions"]
    return sorted(e["reasoner_id"] for e in ex)

sa, sb = shape(a), shape(b)
for x, y in zip(sa, sb):
    print(f"{x:34s} {y}")
print()
print("identical:", sa == sb)

r04_diagnose                       r04_diagnose
r04_lens_blast_scope               r04_lens_blast_scope
r04_lens_change_correlation        r04_lens_change_correlation
r04_lens_dependency_health         r04_lens_dependency_health
r04_lens_resource_contention       r04_lens_resource_contention
r04_lens_timeline                  r04_lens_timeline
r04_synthesize                     r04_synthesize

identical: True


That is the strength. A written-down process is auditable, repeatable, and cheap to
reason about: you know what it will do before you run it.

In [7]:
for label, d in (("inc-008", a), ("inc-011", b)):
    r = d["result"]
    print(f"--- {label}  (confident={r['confident']})")
    print("root cause:", r["root_cause"])
    print()

--- inc-008  (confident=True)
root cause: Deploy dep-2201 introduced a metrics hook in notification-worker that registers a listener per render call on the module-level EventEmitter, causing a memory leak of unremoved listeners that accumulates until heap exhaustion triggers OOMKilled restarts.

--- inc-011  (confident=True)
root cause: The hypervisor maintenance dep-7051 live-migrated VMs on node eu-c1-n07, causing a 92.5-second clock drift in chronyd, which led to JWT 'token used before issued' errors and a 401 rate spike exclusively on pod booking-api-77bd-c3.



And that is also the limit. Both answers happen to be right, but `inc-011` turns on a
clock that drifted, and no clock lens exists — `resource_contention` stumbled into the
evidence. The process cannot ask a question nobody wrote down; it can only get lucky.


## What you learned

- **Fan-out is `asyncio.gather` over `router.app.call`** — five parallel children, one merge, one DAG.
- **A fixed graph has zero process variance.** Its shape is code, so it is the same on every input.
- **It can only ask the questions you anticipated.** The lens that would have cracked `inc-011` was never on the list.

**Next:** 05 · the graph the model writes — same fan-out, but the questions are composed at runtime, for this incident.